# 01. Data Exploration and Visualization — GSE287331

**AGGIUNGERE DESCRIZIONE**


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     01-data-exploration-visualization-GSE287331        ║
# ║ Description:  MANCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA       ║
# ║               MANCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA       ║
# ║               MANCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA       ║
# ║ Dataset(s):   GSE287331                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 13-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


### Libraries

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super


In [1]:
# ============================================
# INTRA-DATASET EXPLORATION & VISUALIZATION — ONE-CELL PIPELINE
# Robust UMAP + Robust EPIC Manifest Coverage (auto header or skiprows=6)
# DNB-like network plots (overall + per-group with same DNB CpGs)
# ============================================

from __future__ import annotations
import os, io, glob, json, time, shutil, inspect, warnings, gzip, bz2
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import polars as pl
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.manifold import TSNE
from sklearn.utils import validation as skval
import networkx as nx


# ---------------------
# USER SETTINGS (EDIT)
# ---------------------
DATASET_NAME  = "GSE287331"
OUTPUT_DIR    = Path(f"./EXPLORATION_{DATASET_NAME}")

# Input paths
PATH_BETA     = Path("/kaggle/input/gse287331-parquet/GSE287331.parquet")  # samples×CpGs, already oriented
PATH_PHENO    = Path("/kaggle/input/pheno-gse287331/pheno_GSE287331.parquet")

# Column names now standardized across all artifacts
BETA_ID_COL    = "id_tissue"  # sample identifier column in β Parquet/CSV
BETA_LABEL_COL = "label"      # optional; main label comes from pheno but we drop from β if present

# OPTIONAL: Illumina manifest / annotation (EPIC v1.0 B5)
PATH_MANIFEST = Path("/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv")
ID_REF_COL    = "ID_REF"        # when loading Series Matrix (CpG×sample) — not used for Parquet here
ORIENT        = "sample_by_cpg"  # {"sample_by_cpg", "cpg_by_sample"}

# Plot style
USE_TEX       = True   # auto-fallback if LaTeX not available
SEED          = 42

# Subsampling / sizes
MAX_DENSITY_POINTS  = 300_000
MAX_RIDGE_POINTS    = 300_000
MAX_DENSITY_SAMPLES = 48
TOP_OUTLIER_CPGS    = 200
PCA_TOP_CPGS        = 20_000
IPCA_COMPONENTS     = 100

# Thresholds (flagging only; NO filtering here)
SAMPLE_NA_FLAG      = 0.03
CPG_NA_FLAG         = 0.10
ROBUST_Z_CUTOFF     = 3.5
DELTA_THRESHOLDS    = (0.1, 0.2)

# Groups (label int -> name)
LABEL_MAP      = {0:"Normal", 1:"Adjacent", 2:"Tumor"}
GROUP_ORDER    = ["Normal","Adjacent","Tumor"]
LEGEND_TEXT    = {"Normal":"Normal Samples", "Adjacent":"Adjacent Samples", "Tumor":"Tumor Samples"}
LEGEND_ORDER   = [LEGEND_TEXT[g] for g in GROUP_ORDER]

# Global plotting toggles
ADD_TITLES    = False
SHOW_PLOTS    = False

# Global plot counter
PLOT_ID = 1

# =====================
# UTILITIES & LOGGING
# =====================

def seed_everything(seed:int=SEED):
    np.random.seed(seed)

def ensure_dirs():
    for sub in ["figures", "tables", "logs"]:
        (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

def save_json(obj:dict, path:Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def info(msg: str): print(f"[INFO] {msg}")
def ok(msg: str):   print(f"[OK]   {msg}")
def warn(msg: str): print(f"[WARN] {msg}")

# ---------------------------
# LaTeX-safe text
# ---------------------------

def _has_latex() -> bool:
    if not USE_TEX:
        return False
    return (
        shutil.which("pdflatex") is not None or
        shutil.which("xelatex") is not None or
        shutil.which("lualatex") is not None
    )

def L(text: str | None) -> str | None:
    if text is None: return None
    t = str(text)
    if not _has_latex():
        return (t.replace("Δβ","Delta beta").replace("β","beta").replace("Δ","Delta")
                 .replace("μ","mu").replace("≤","<=").replace("≥",">=")
                 .replace("×","x").replace("–","-").replace("—","-").replace("…","..."))
    for k,v in {"Δβ": r"$\Delta\beta$","β": r"$\beta$","Δ": r"$\Delta$","δ": r"$\delta$","μ": r"$\mu$",
                "≤": r"$\leq$","≥": r"$\geq$","×": r"$\times$"}.items():
        t = t.replace(k,v)
    t = t.replace("–","--").replace("—","---").replace("…",r"\ldots{}")
    specials = {"\\": r"\textbackslash{}","&": r"\&","%": r"\%","$": r"\$","#": r"\#","_": r"\_",
                "{": r"\{","}": r"\}","~": r"\textasciitilde{}","^": r"\textasciicircum{}"}
    out, in_math = [], False
    for ch in t:
        if ch == "$":
            out.append("$"); in_math = not in_math; continue
        if not in_math and ch in specials: out.append(specials[ch])
        else: out.append(ch)
    return "".join(out)

# --- Plotting style ---

def apply_thesis_style(use_tex: bool = True,
                       legend_position: str = "upper right",
                       legend_outside: bool = False,
                       set_color_cycle: bool = True):
    """
    Uniform thesis plotting style.
    - Safe LaTeX: enables text.usetex only if TeX is available.
    - Optional global color cycle (viridis band).
    - Exposes place_legend(), VIRIDIS, and GROUP_COLORS.
    - Honors THESIS_FIGSIZE if defined in globals().
    """
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    def _has_latex():
        import shutil
        return shutil.which("latex") is not None

    # Normalize legend position
    def _normalize_pos(pos: str) -> str:
        key = (pos or "upper right").strip().lower().replace("top", "upper").replace("bottom","lower")
        return {
            "upper right":"upper right","upper left":"upper left",
            "lower right":"lower right","lower left":"lower left",
            "center":"center","best":"best"
        }.get(key, "upper right")

    default_loc = _normalize_pos(legend_position)

    # Figure size: prefer THESIS_FIGSIZE if present
    figsize = globals().get("THESIS_FIGSIZE", (6.8, 4.5))

    # Base theme
    sns.set_theme(style="whitegrid", context="notebook")
    tex_flag = bool(use_tex and _has_latex())

    # Optional color cycle (viridis band with pleasant mid-tones)
    viridis_cycle = mpl.cm.viridis(np.linspace(0.15, 0.85, 8))
    if set_color_cycle:
        sns.set_palette(viridis_cycle)

    mpl.rcParams.update({
        # Typography
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman", "Latin Modern Roman", "Times New Roman"],
        "mathtext.fontset": "cm",
        "text.usetex": tex_flag,
        "pdf.fonttype": 42, "ps.fonttype": 42,

        # Size
        "figure.figsize": figsize,
        "font.size": 8.5,
        "axes.labelsize": 10.5,
        "xtick.labelsize": 9, "ytick.labelsize": 9,
        "legend.fontsize": 9.5, "legend.title_fontsize": 9.5,

        # Axes look
        "axes.grid": True, "grid.alpha": 0.25,
        "axes.facecolor": "white",
        "axes.edgecolor": "#D0D0D0",
        "axes.linewidth": 0.8,

        # Legend defaults
        "legend.frameon": True,
        "legend.facecolor": "white",
        "legend.edgecolor": "#D0D0D0",
        "legend.loc": default_loc,
        "legend.framealpha": 1.0,
        "legend.handlelength": 1.8,
        "legend.handletextpad": 0.6,
        "legend.borderpad": 0.4,
    })
    if set_color_cycle:
        mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=viridis_cycle)

    # legend helper
    def place_legend(ax=None, position: str | None = None,
                     outside: bool | None = None, fontsize: float | None = None):
        if ax is None:
            ax = plt.gca()

        loc_val = default_loc if position is None else _normalize_pos(position)
        outside_flag = (legend_outside if outside is None else bool(outside))

        kw = dict(
            loc=loc_val,
            frameon=True,
            facecolor="white",
            edgecolor=mpl.rcParams.get("legend.edgecolor", "#D0D0D0"),
            framealpha=1.0,
            fontsize=mpl.rcParams["legend.fontsize"] if fontsize is None else fontsize,
            handlelength=1.8,
            handletextpad=0.6,
            borderpad=0.4,
            fancybox=False,
        )

        if outside_flag:
            # mappa loc → posizione reale fuori dagli assi
            anchor_map = {
                "upper right": (1.02, 1.0),
                "lower right": (1.02, 0.0),
                "upper left":  (-0.02, 1.0),
                "lower left":  (-0.02, 0.0),
                "center":      (0.5, -0.1),
                "best":        (1.02, 1.0),
            }
            anchor = anchor_map.get(loc_val, (1.02, 1.0))
            kw.update({"bbox_to_anchor": anchor, "borderaxespad": 0.0})

        leg = ax.legend(**kw)
        if leg is not None:
            leg.get_frame().set_linewidth(0.8)
            leg.get_frame().set_edgecolor(mpl.rcParams.get("legend.edgecolor", "#D0D0D0"))
            leg.get_frame().set_facecolor("white")
        return ax


    # Export helpers/globals for the rest of the notebook
    globals()["place_legend"] = place_legend
    globals()["VIRIDIS"] = mpl.cm.viridis

    # Safe GROUP_COLORS (falls back if GROUP_ORDER missing)
    group_order = globals().get("GROUP_ORDER", ["Normal", "Adjacent", "Tumor"])
    colors = VIRIDIS(np.linspace(0.15, 0.85, len(group_order)))
    globals()["GROUP_COLORS"] = {grp: col for grp, col in zip(group_order, colors)}


def set_title(text: str):
    if ADD_TITLES: plt.title(L(text))

def savefig(name:str):
    global PLOT_ID
    fig = plt.gcf()
    for ax in fig.axes:
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_linewidth(0.8); spine.set_edgecolor(mpl.rcParams.get("axes.edgecolor","#000"))
    out_name = f"{DATASET_NAME}_{PLOT_ID:02d}_{name}.pdf"
    out = OUTPUT_DIR/"figures"/out_name
    plt.tight_layout()
    plt.savefig(out, bbox_inches="tight")
    if SHOW_PLOTS: plt.show()
    plt.close()
    ok(f"Saved figure → {out}")
    PLOT_ID += 1

def legend_label(g: str) -> str:
    return LEGEND_TEXT.get(g, g)

# =====================
# DATA LOADING
# =====================

def load_beta_matrix(path:Path, orient:str="sample_by_cpg", id_ref_col:str="ID_REF") -> pd.DataFrame:
    t0 = time.time()
    ext = path.suffix.lower()
    if ext == ".parquet":
        lf = pl.scan_parquet(str(path))
        df_pl = lf.collect()
        if BETA_ID_COL not in df_pl.columns:
            raise KeyError(f"Column '{BETA_ID_COL}' not found in beta Parquet. Available: {df_pl.columns}")
        df_pl = df_pl.rename({BETA_ID_COL: "id_tissue"})
        if BETA_LABEL_COL in df_pl.columns:
            df_pl = df_pl.with_columns(pl.col(BETA_LABEL_COL).cast(pl.Int64))
        exclude_cols = ["id_tissue"]
        if BETA_LABEL_COL in df_pl.columns: exclude_cols.append(BETA_LABEL_COL)
        df_pl = df_pl.with_columns(pl.all().exclude(exclude_cols).cast(pl.Float32))
        df_pd = df_pl.to_pandas()
        cpglike_cols = [c for c in df_pd.columns if c not in ["id_tissue", BETA_LABEL_COL]]
        beta = df_pd.set_index("id_tissue")[cpglike_cols]
        ok(f"β Parquet (Polars) loaded → shape={beta.shape} in {time.time()-t0:0.2f}s")
        return beta
    elif ext in {".csv",".txt",".tsv"}:
        sep = "," if ext==".csv" else ("\t" if ext in {".tsv",".txt"} else None)
        df = pd.read_csv(path, sep=sep)
        if BETA_ID_COL in df.columns: df = df.set_index(BETA_ID_COL)
        elif "id_tissue" in df.columns: df = df.set_index("id_tissue")
        else: warn("No explicit sample ID column in beta CSV; using index as id_tissue."); df.index.name = "id_tissue"
        # drop label column if present
        if BETA_LABEL_COL in df.columns:
            df = df.drop(columns=[BETA_LABEL_COL])
        df = df.apply(pd.to_numeric, errors="coerce").astype(np.float32)
        ok(f"β text loaded & cast (shape={df.shape}) in {time.time()-t0:0.2f}s")
        return df
    else:
        raise ValueError(f"Unsupported file extension for beta matrix: {ext}")

def load_pheno(path:Path) -> pd.DataFrame:
    t0 = time.time()
    ext = path.suffix.lower()
    if ext == ".parquet":
        lf = pl.scan_parquet(str(path))
        ph_pl = lf.collect()
        ph = ph_pl.to_pandas()
    else:
        ph = pd.read_csv(path)
    ph.columns = [c.strip() for c in ph.columns]
    required = {"id_tissue","label"}
    missing = required - set(ph.columns)
    if missing: raise KeyError(f"Pheno missing required columns: {missing}")
    ok(f"pheno loaded (rows={len(ph)}, cols={len(ph.columns)}) in {time.time()-t0:0.2f}s")
    return ph

def align_beta_pheno(beta:pd.DataFrame, ph:pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    if beta.index.name is None or str(beta.index.name).lower() not in {"sample","id_tissue"}:
        beta.index.name = "id_tissue"
    ph = ph.copy()
    ph["label"] = pd.to_numeric(ph["label"], errors="coerce").astype("Int64")
    common = beta.index.intersection(ph["id_tissue"])
    beta = beta.loc[common].copy()
    ph   = ph.set_index("id_tissue").loc[common].copy()
    labels = ph["label"].map(LABEL_MAP)
    counts = labels.value_counts(dropna=False)
    ok("Aligned β↔pheno: samples={} | groups: {}".format(
       len(common), ", ".join([f"{g}:{counts.get(g,0)}" for g in GROUP_ORDER])
    ))
    return beta, labels, ph

# =====================
# SNAPSHOT & MISSINGNESS
# =====================

def snapshot_and_missing(beta:pd.DataFrame, labels:pd.Series) -> dict:
    n_samples, n_cpg = beta.shape
    miss_sample = beta.isna().mean(axis=1)
    miss_cpg    = beta.isna().mean(axis=0)
    (OUTPUT_DIR/"tables"/"missing_per_sample.csv").write_text(
        miss_sample.to_frame("missing_frac").to_csv(index=True)
    )
    (OUTPUT_DIR/"tables"/"missing_per_cpg.csv").write_text(
        miss_cpg.to_frame("missing_frac").to_csv(index=True)
    )
    fig = new_fig()
    sns.histplot(miss_sample, bins=30, color=VIRIDIS(0.7), kde=False, label=L("Missing fraction (Samples)"))
    plt.xlabel(L("Missing fraction per Sample")); plt.ylabel(L("Number of Samples")); plt.xlim(0.0,1.0)
    place_legend(position="upper right"); savefig("missing_per_sample")

    fig = new_fig()
    sns.histplot(miss_cpg, bins=30, color=VIRIDIS(0.7), kde=False, label=L("Missing fraction (CpGs)"))
    plt.xlabel(L("Missing fraction per CpG")); plt.ylabel(L("Number of CpGs")); plt.xlim(0.0,1.0)
    place_legend(position="upper right"); savefig("missing_per_cpg")

    summary = {
        "n_samples": int(n_samples), "n_cpg": int(n_cpg),
        "missing_sample_median": float(np.nanmedian(miss_sample)),
        "missing_sample_iqr": float(np.nanpercentile(miss_sample,75)-np.nanpercentile(miss_sample,25)),
        "missing_cpg_median": float(np.nanmedian(miss_cpg)),
        "missing_cpg_iqr": float(np.nanpercentile(miss_cpg,75)-np.nanpercentile(miss_cpg,25)),
        "n_samples_flagged_missing": int((miss_sample >= SAMPLE_NA_FLAG).sum()),
        "n_cpg_flagged_missing": int((miss_cpg >= CPG_NA_FLAG).sum()),
    }
    if summary["missing_sample_median"]==0.0 and summary["missing_cpg_median"]==0.0:
        ok("Missingness: none detected at median level (both per-sample and per-CpG are 0.0)")
    else:
        info(f"Missingness — per-sample median={summary['missing_sample_median']:.4f}, per-CpG median={summary['missing_cpg_median']:.4f}")
    info(f"Flags — samples≥{SAMPLE_NA_FLAG*100:.1f}%: {summary['n_samples_flagged_missing']} | CpGs≥{CPG_NA_FLAG*100:.1f}%: {summary['n_cpg_flagged_missing']}")
    return summary

# =====================
# DISTRIBUTIONS
# =====================

def _collect_group_betas(beta: pd.DataFrame, labels: pd.Series, group: str,
                         max_points: int = MAX_DENSITY_POINTS) -> np.ndarray:
    idx = labels[labels == group].index
    if len(idx) == 0: return np.array([], dtype=np.float32)
    arr = beta.loc[idx].to_numpy(dtype=np.float32, copy=False).ravel()
    arr = arr[np.isfinite(arr)]
    if max_points is not None and arr.size > max_points:
        rng = np.random.default_rng(SEED)
        arr = rng.choice(arr, size=max_points, replace=False)
    return arr

# ================================
# DENSITY PLOTS — NEW VARIANTS
# ================================
from __future__ import annotations
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Assumptions:
# - beta: DataFrame shaped (n_samples, n_cpgs) with sample_id as index
# - labels: Series indexed like beta.index with values in GROUP_ORDER
# - Existing helpers/constants available in scope:
#   L(), place_legend(position="upper right"), savefig(name),
#   GROUP_ORDER, GROUP_COLORS, legend_label(g), MAX_DENSITY_POINTS

def _subsample_vals(vals: np.ndarray, max_points: int) -> np.ndarray:
    """Return up to max_points finite values from vals, uniformly at random."""
    vals = vals[np.isfinite(vals)]
    if vals.size <= max_points:
        return vals
    rng = np.random.default_rng(42)
    idx = rng.choice(vals.size, size=max_points, replace=False)
    return vals[idx]

def beta_density_random_samples(
    beta: pd.DataFrame,
    labels: pd.Series,
    n_samples: int = 15,
    max_points: int = None,
    seed: int = 42,
    legend_loc: str = "upper right",
):
    """
    Plot KDE curves for ~n_samples randomly chosen samples.
    Colors follow the viridis colormap.
    Legend shows ONE viridis-colored example curve.
    """
    if max_points is None:
        max_points = MAX_DENSITY_POINTS

    # Align indices
    common_idx = beta.index.intersection(labels.index)
    beta = beta.loc[common_idx]
    labels = labels.loc[common_idx]

    # Random samples
    rng = np.random.default_rng(seed)
    n_pick = min(n_samples, beta.shape[0])
    picked = rng.choice(beta.index.values, size=n_pick, replace=False)

    # --- Viridis colors
    # sample n_pick distinct values from the viridis colormap
    cmap = plt.cm.get_cmap("viridis")
    colors = cmap(np.linspace(0.15, 0.95, n_pick))  # avoid too-dark/too-light ends

    # --- Figure
    fig = new_fig()
    for i, sid in enumerate(picked):
        vals = beta.loc[sid].to_numpy(dtype=float, copy=False)
        vals = _subsample_vals(vals, max_points)
        if vals.size == 0:
            continue
        sns.kdeplot(
            vals,
            linewidth=1.2,
            fill=False,
            color=colors[i],
            alpha=0.9,
            label=None,
            clip=(-0.05, 1.05),
        )

    # --- Labels
    plt.xlabel(L(r"$\beta$-value"))
    plt.ylabel(L("Density"))
    plt.xlim(-0.1, 1.1)

    # --- Legend (single viridis example curve)
    from matplotlib.lines import Line2D
    example_color = cmap(0.6)  # nice mid-viridis green
    handle = Line2D(
        [], [], 
        color=example_color, 
        lw=1.5, 
        label=L("Example β-value curves (Random Samples)")
    )
    plt.legend(handles=[handle], loc=legend_loc, frameon=True)

    # --- Save
    plot_name = "beta_density_random_samples"
    savefig(plot_name)




def _kde_on_grid(values: np.ndarray, grid: np.ndarray, bw_method=None) -> np.ndarray:
    """Compute Gaussian KDE on a fixed grid for given values in [0,1]."""
    values = values[np.isfinite(values)]
    if values.size < 5:
        return np.zeros_like(grid)
    try:
        kde = stats.gaussian_kde(values, bw_method=bw_method)
        y = kde(grid)
        # Normalize to unit area over the grid step so each curve integrates to ~1
        dx = np.diff(grid).mean() if grid.size > 1 else 1.0
        area = (y.sum() * dx)
        if area > 0:
            y = y / area
        return y
    except Exception:
        return np.zeros_like(grid)


def beta_density_group_mean(
    beta: pd.DataFrame,
    labels: pd.Series,
    groups=None,
    grid_size: int = 256,
    max_points_per_sample: int = None,
    bw_method=None,
    legend_loc: str = "upper right",
):
    """
    Plot the mean KDE per group (Normal/Adjacent/Tumor). For each group:
      - compute per-sample KDE on a common grid in [0,1]
      - average the curves across samples in that group
    The result is a smooth “mean density” per group, not a pooled KDE.
    """
    if groups is None:
        groups = GROUP_ORDER
    if max_points_per_sample is None:
        max_points_per_sample = MAX_DENSITY_POINTS

    # Align indices
    common_idx = beta.index.intersection(labels.index)
    beta = beta.loc[common_idx]
    labels = labels.loc[common_idx]

    grid = np.linspace(0.0, 1.0, grid_size)

    fig = new_fig()
    for g in groups:
        sids = labels.index[labels == g]
        if len(sids) == 0:
            continue

        curves = []
        for sid in sids:
            vals = beta.loc[sid].to_numpy(dtype=float, copy=False)
            vals = _subsample_vals(vals, max_points_per_sample)
            if vals.size == 0:
                continue
            y = _kde_on_grid(vals, grid, bw_method=bw_method)
            if np.any(y):
                curves.append(y)

        if len(curves) == 0:
            continue

        mean_y = np.mean(np.vstack(curves), axis=0)
        plt.plot(
            grid, mean_y, linewidth=2.0,
            label=L(f"{legend_label(g)} — Mean") if callable(legend_label) else L(f"{g} — Mean"),
            color=GROUP_COLORS[g]
        )

    plt.xlabel(L(r"$\beta$-value"))
    plt.ylabel(L("Mean density"))
    plt.xlim(0, 1)
    place_legend(position=legend_loc)

    # --- Thesis-style save ---
    plot_name = "beta_density_group_mean"
    dataset = globals().get("DATASET_NAME", "DATASET")
    fig_idx = globals().get("PLOT_INDEX", 2)  # set PLOT_INDEX before calling if needed
    savefig(f"{plot_name}")


def ridge_by_group(beta: pd.DataFrame, labels: pd.Series):
    rng = np.random.default_rng(SEED); N_PER_GROUP = 15
    for g in GROUP_ORDER:
        idx_g = labels[labels == g].index
        if len(idx_g)==0: continue
        take = min(N_PER_GROUP, len(idx_g))
        chosen = list(rng.choice(idx_g, size=take, replace=False))
        B = beta.loc[chosen]
        fig = new_fig()
        for sid in chosen:
            vals = B.loc[sid].to_numpy(dtype=np.float32); vals = vals[np.isfinite(vals)]
            if vals.size == 0: continue
            sns.kdeplot(vals, linewidth=1.0, alpha=0.6, color=GROUP_COLORS[g])
        plt.plot([], [], linestyle="-", color=GROUP_COLORS[g], label=L(f"Density — {legend_label(g)}"))
        plt.xlabel(L(r"$\beta$-value")); plt.ylabel(L("Density")); plt.xlim(0,1)
        ymin,ymax = plt.ylim(); plt.ylim(ymin, ymax)
        place_legend(position="upper right"); savefig(f"ridge_{g.lower()}")

def mean_beta_by_sample(beta: pd.DataFrame, labels: pd.Series):
    means = beta.mean(axis=1)
    (OUTPUT_DIR/"tables"/"mean_beta_per_sample.csv").write_text(
        means.to_frame("mean_beta").join(labels.rename("group")).to_csv()
    )

    # FIG 1: histogram + density
    fig = new_fig(); ax = plt.gca()
    # colori scambiati: istogramma più chiaro, densità più scura
    bar_color  = plt.cm.viridis(0.7)   # più chiaro
    line_color = plt.cm.viridis(0.35)   # più scuro

    sns.histplot(
        means,
        bins=30,
        stat="count",
        color=bar_color,
        alpha=0.85,
        edgecolor=None,
        linewidth=0,
        ax=ax,
    )
    sns.kdeplot(
        means,
        bw_adjust=0.9,
        linewidth=1.8,
        color=line_color,
        ax=ax,
    )

    ax.set_xlabel(L("Average Methylation ($\\beta$)"))
    ax.set_ylabel(L("Number of Samples"))

    bar_handle = mpl.patches.Patch(
        facecolor=bar_color,
        edgecolor=None,
        label=L("Histogram of Sample Mean $\\beta$")
    )
    line_handle = mpl.lines.Line2D(
        [], [],
        color=line_color,
        lw=1.8,
        label=L("Density of Sample Mean $\\beta$")
    )
    ax.legend(handles=[bar_handle, line_handle], loc="upper right", frameon=True)
    savefig("mean_beta_per_sample")

    # FIG 2: distribution by group (unchanged)
    df = pd.DataFrame({"mean_beta": means, "group": labels}).dropna()
    fig = new_fig(); ax2 = plt.gca()
    group_colors = {
        g: c for g, c in zip(GROUP_ORDER, plt.cm.viridis(np.linspace(0.15, 0.85, 3)))
    }
    sns.boxplot(
        data=df,
        x="group",
        y="mean_beta",
        order=GROUP_ORDER,
        showcaps=False,
        showfliers=False,
        boxprops={"facecolor": "none"},
        ax=ax2,
    )
    sns.stripplot(
        data=df,
        x="group",
        y="mean_beta",
        order=GROUP_ORDER,
        alpha=0.75,
        dodge=False,
        size=4,
        palette=[group_colors[g] for g in GROUP_ORDER],
        ax=ax2,
    )
    ax2.set_xlabel("")
    ax2.set_ylabel(L("Per-Sample Mean $\\beta$"))

    handles = [
        mpl.lines.Line2D(
            [], [],
            color=group_colors[g],
            marker="o",
            linestyle="none",
            markersize=5,
            label=L(legend_label(g)),
        )
        for g in GROUP_ORDER
        if (df["group"] == g).any()
    ]
    ax2.legend(handles=handles, loc="lower left", frameon=True)
    savefig("mean_beta_distribution_by_group")


# =====================
# SAMPLE-LEVEL QC (OUTLIERS)
# =====================

def outlier_burden(beta:pd.DataFrame, labels:pd.Series) -> pd.DataFrame:
    ref_idx = labels[labels=="Normal"].index
    REF = beta.loc[ref_idx] if len(ref_idx)>0 else beta
    ref_med = REF.median(axis=0)
    ref_mad = (REF - ref_med).abs().median(axis=0) * 1.4826
    Z = (beta - ref_med) / ref_mad.replace(0, np.nan)
    outlier_mask = Z.abs() > ROBUST_Z_CUTOFF
    outlier_count = outlier_mask.sum(axis=1)
    df = pd.DataFrame({"group":labels, "outliers":outlier_count}).dropna()
    df["log10_outliers"] = np.log10(df["outliers"].replace(0, np.nan))
    df.to_csv(OUTPUT_DIR/"tables"/"outlier_burden_per_sample.csv")
    fig = new_fig(); ax = plt.gca()
    sns.boxplot(data=df, x="group", y="log10_outliers", order=GROUP_ORDER, showcaps=False, showfliers=False,
                boxprops={'facecolor':'none'}, linewidth=0.9, palette=[GROUP_COLORS[g] for g in GROUP_ORDER])
    sns.stripplot(data=df, x="group", y="log10_outliers", order=GROUP_ORDER, alpha=0.7, dodge=False,
                  palette=[GROUP_COLORS[g] for g in GROUP_ORDER])
    plt.xlabel(""); plt.ylabel(L(r"$\log_{10}(\mathrm{outlier\ burden})$"))
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=GROUP_COLORS[g], edgecolor='none', label=L(f"Outlier burden — {legend_label(g)}"))
               for g in GROUP_ORDER if (df["group"]==g).any()]
    leg = ax.legend(handles=handles, loc="upper left", frameon=True, fontsize=mpl.rcParams["legend.fontsize"])
    leg.get_frame().set_linewidth(0.8)
    savefig("outlier_burden")
    head5 = df.sort_values("outliers", ascending=False).head(5)
    info("Top-5 Samples by outlier burden:" + "".join([f"\n  - {idx}: {row.outliers}" for idx, row in head5.iterrows()]))
    return df

# =====================
# CPG-LEVEL EXPLORATION (HEATMAP)
# =====================

def cpg_outlier_heatmap_numbered(
    beta: pd.DataFrame,
    k_iqr: float = 3.0,
    top_n: int = None,
    cmap: str = "viridis",
    cbar_label: str = r"$\beta$-value",
    outfile: str = "heatmap_top_outlier_cpgs_numbered",
    max_xticks: int = 12,
    max_yticks: int = 12,
    ticklabelsize: int = 8,
    draw_quartile_rules: bool = True,
):
    """
    Heatmap of top CpG sites ranked by outlier burden (k*IQR), thesis style.

    beta: DataFrame of shape (n_samples × n_cpgs), values in [0, 1] or NaN.
    Rows = samples, columns = CpG sites (ranked by outlier burden).
    """
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import pandas as pd
    from warnings import warn
    from pathlib import Path

    # ===== Helper presi dai globals, con fallback locali =====
    out_dir = globals().get("OUTPUT_DIR", Path("."))
    new_fig_fn = globals().get("new_fig", None)
    place_legend_fn = globals().get("place_legend", None)
    savefig_fn = globals().get("savefig", None)
    info_fn = globals().get("info", None)
    L_fn = globals().get("L", None)
    apply_thesis_fn = globals().get("apply_thesis", None)

    if new_fig_fn is None:
        def new_fig_fn(aspect_ratio=0.6, width_pt=469.75):
            plt.figure()
            return plt.gcf()

    if place_legend_fn is None:
        def place_legend_fn(ax, position="upper right"):
            ax.legend(loc=position)

    if savefig_fn is None:
        def savefig_fn(path):
            plt.savefig(f"{path}.png", bbox_inches="tight")
            print(f"Saved {path}.png")

    if info_fn is None:
        info_fn = print

    if L_fn is None:
        L_fn = lambda x: x

    if apply_thesis_fn is None:
        def apply_thesis_fn(*args, **kwargs):
            return

    # ------- Safety checks -------
    if beta is None or beta.size == 0 or beta.shape[0] == 0 or beta.shape[1] == 0:
        warn("[heatmap] Empty β matrix — skipping plot.")
        return

    beta = beta.apply(pd.to_numeric, errors="coerce").clip(lower=0.0, upper=1.0)

    if top_n is None:
        top_n = globals().get("TOP_OUTLIER_CPGS", 500)
    top_n = max(1, min(top_n, beta.shape[1]))

    # ------- 1) Outlier burden per CpG using k*IQR -------
    Q1 = beta.quantile(0.25, numeric_only=True)
    Q3 = beta.quantile(0.75, numeric_only=True)
    IQR = Q3 - Q1
    lower = Q1 - k_iqr * IQR
    upper = Q3 + k_iqr * IQR

    outliers = (beta.lt(lower, axis=1)) | (beta.gt(upper, axis=1))
    counts = outliers.sum(axis=0).sort_values(ascending=False)

    (out_dir / "tables").mkdir(parents=True, exist_ok=True)
    counts.to_frame("outlier_sample_count").to_csv(
        out_dir / "tables" / "cpg_outlier_counts.csv"
    )

    # ------- 2) Select top CpGs and build samples × K matrix -------
    top_cpgs = counts.head(top_n).index
    if len(top_cpgs) == 0:
        warn("[heatmap] No CpGs selected (all counts=0?) — skipping plot.")
        return

    mat = beta[top_cpgs]      # samples × K
    n_samples, K = mat.shape

    # ------- 3) Figure & main axis (thesis style) -------
    fig = new_fig_fn()
    ax = plt.gca()

    hm = sns.heatmap(
        mat,
        ax=ax,
        cmap=cmap,
        vmin=0.0,
        vmax=1.0,
        cbar=False,      # no automatic colorbar
        yticklabels=False,
        xticklabels=False,
        rasterized=True,
    )

    # Applica lo stile tesi (senza legenda automatica)
    try:
        apply_thesis_fn(ax=ax)
    except TypeError:
        try:
            apply_thesis_fn(ax, legend=False)
        except TypeError:
            pass
        # ============================================================
    # 4) Usa place_legend per avere la bbox, poi mettiamo il box
    #    FISSO quasi nell'angolo in alto a destra
    # ============================================================
    dummy_line = mpl.lines.Line2D(
        [], [], color=mpl.cm.get_cmap(cmap)(0.6), label=L_fn(r"$\beta$-value")
    )
    ax.add_line(dummy_line)

    place_legend_fn(ax=ax, position="upper right")
    leg = ax.get_legend()

    # serve per avere la dimensione di riferimento
    fig.canvas.draw()
    bbox_pixels = leg.get_frame().get_window_extent()
    bbox_fig = bbox_pixels.transformed(fig.transFigure.inverted())
    x0, y0, w, h = bbox_fig.bounds

    leg.remove()
    dummy_line.remove()

    # ------------------------------------------------------------
    # 5) Box QUASI fuori: in alto a destra, più stretto e un po’ più alto
    # ------------------------------------------------------------
    margin = 0.01                 # distanza minima dal bordo figura
    new_w = w * 0.85              # leggermente più stretto
    new_h = h * 1.25              # un po' più alto

    # posiziona il box "al minimo prima di stare fuori"
    new_x = 1.0 - new_w - margin -0.03  # quasi appiccicato a destra
    new_y = 1.0 - new_h - margin -0.04 # quasi appiccicato in alto

    cax = fig.add_axes([new_x, new_y, new_w, new_h])

    edge_col = mpl.rcParams.get("legend.edgecolor", "#D0D0D0")
    face_col = mpl.rcParams.get("legend.facecolor", "white")

    cax.set_facecolor(face_col)
    for spine in cax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(edge_col)
        spine.set_linewidth(0.8)

    cax.set_xlim(0.0, 1.0)
    cax.set_ylim(0.0, 1.0)
    cax.set_xticks([])
    cax.set_yticks([])
    cax.tick_params(axis="both", bottom=False, top=False,
                    left=False, right=False, length=0)

    # Barra viridis: larga tutto il box (niente bianco a sinistra)
    gradient = np.linspace(0.0, 1.0, 256).reshape(1, -1)
    cax.imshow(
        gradient,
        aspect="auto",
        cmap=cmap,
        origin="lower",
        extent=[0.0, 1.0, 0.45, 1.0],  # tutta la larghezza, parte alta del box
    )

    font_size = mpl.rcParams.get("legend.fontsize", 9.5)
    font_color = "black"

    # NIENTE scritta β-value

    # 0 quasi al bordo sinistro in basso
    cax.text(
        0.0,
        0.20,          # quasi al bordo inferiore ma dentro
        r"$0$",
        fontsize=font_size,
        color=font_color,
        ha="left",
        va="center",
        transform=cax.transAxes,
    )

    # 1 quasi al bordo destro in basso
    cax.text(
        1.0,
        0.20,
        r"$1$",
        fontsize=font_size,
        color=font_color,
        ha="right",
        va="center",
        transform=cax.transAxes,
    )

    cax.set_xlabel("")
    cax.set_ylabel("")

    

    # ------- 6) Axis labels per la heatmap -------
    ax.set_xlabel(L_fn("CpG sites (ranked by outlier burden)"))
    ax.set_ylabel(L_fn("Samples"))

    # ------- 7) Smart ticks (indices 1..K, 1..N) -------
    def _nice_ticks(n_items: int, max_ticks: int):
        if n_items <= 0:
            return [], []
        n_ticks = min(max_ticks, n_items)
        pos = np.linspace(0, n_items - 1, num=n_ticks, dtype=int)
        lab = (pos + 1).astype(int)
        return pos + 0.5, lab

    xlocs, xlabels = _nice_ticks(K, max_xticks)
    ax.set_xticks(xlocs)
    ax.set_xticklabels(
        [str(x) for x in xlabels],
        fontsize=ticklabelsize,
        rotation=0,
    )

    ylocs, ylabels = _nice_ticks(n_samples, max_yticks)
    ax.set_yticks(ylocs)
    ax.set_yticklabels(
        [str(y) for y in ylabels],
        fontsize=ticklabelsize,
        rotation=0,
    )

    # ------- 8) Optional vertical guidelines at CpG quartiles -------
    if draw_quartile_rules and K >= 8:
        for q in (0.25, 0.50, 0.75):
            x = (K * q) + 0.5
            ax.axvline(
                x=x,
                color="yellow",
                lw=0.8,
                ls="--",
                alpha=0.9,
                zorder=3,
            )

    # ------- 9) Save -------
    savefig_fn(outfile)

    # ------- 10) Log top-5 -------
    head5 = "\n".join(
        [f"  - {cpg}: {int(cnt)}" for cpg, cnt in counts.head(5).items()]
    )
    info_fn("Top-5 CpGs by outlier-sample count:\n" + head5)

# =====================
# BASIC BIOLOGICAL CONTRASTS (Δβ)
# =====================

def delta_beta_distributions(beta: pd.DataFrame, labels: pd.Series) -> dict:
    """
    Single overlaid Δβ plot with two KDE curves (Tumor–Normal, Adjacent–Normal).
    Hypo/Hyper labels placed at the same original manual positions.
    """
    grp = labels
    results = {}

    def mean_by(group_name: str):
        idx = grp[grp == group_name].index
        return beta.loc[idx].mean(axis=0) if len(idx) > 0 else None

    tum, nor, adj = mean_by("Tumor"), mean_by("Normal"), mean_by("Adjacent")

    viridis_map = VIRIDIS(np.linspace(0.15, 0.85, 5))
    color_tumor    = viridis_map[-1]
    color_adjacent = viridis_map[2]

    deltas = {}

    if tum is not None and nor is not None:
        d_tn = (tum - nor).dropna()
        deltas["Tumor"] = (d_tn, color_tumor, r"$\Delta\beta$ (Tumor - Normal)")
        stats = {**{f"abs≥{t}": int((d_tn.abs() >= t).sum()) for t in DELTA_THRESHOLDS},
                 **{f"frac_abs≥{t}": float((d_tn.abs() >= t).mean()) for t in DELTA_THRESHOLDS},
                 "mean_abs_delta": float(d_tn.abs().mean())}
        save_json(stats, OUTPUT_DIR / "tables" / "delta_tumor_vs_normal_counts.json")
        results["Tumor_vs_Normal"] = stats

    if adj is not None and nor is not None:
        d_an = (adj - nor).dropna()
        deltas["Adjacent"] = (d_an, color_adjacent, r"$\Delta\beta$ (Adjacent - Normal)")
        stats2 = {**{f"abs≥{t}": int((d_an.abs() >= t).sum()) for t in DELTA_THRESHOLDS},
                  **{f"frac_abs≥{t}": float((d_an.abs() >= t).mean()) for t in DELTA_THRESHOLDS},
                  "mean_abs_delta": float(d_an.abs().mean())}
        save_json(stats2, OUTPUT_DIR / "tables" / "delta_adjacent_vs_normal_counts.json")
        results["Adjacent_vs_Normal"] = stats2

    if not deltas:
        return results

    # ----------------------------------------------------------
    # FIGURE (single)
    # ----------------------------------------------------------
    fig = new_fig()

    # Plot both KDE curves
    for key, (vec, color, label_str) in deltas.items():
        sns.kdeplot(
            vec, linewidth=2, fill=True, alpha=0.35,
            color=color, label=L(label_str)
        )

    plt.axvline(0, ls="--", lw=1, color="black")
    plt.xlim(-1.0, 1.0)
    plt.xlabel(L(r"$\Delta\beta$ (Group - Normal)"))
    plt.ylabel(L("Density"))

    # Keep y-limit wide enough for the original positions
    ymin, ymax = plt.ylim()
    # Must reach at least y=5.0 to place both labels exactly where you want
    if ymax < 6.0:
        plt.ylim(0.0, 6.0)

    # ----------------------------------------------------------
    # LABEL POSITIONS — EXACTLY AS IN ORIGINAL CODE
    # ----------------------------------------------------------

    # Tumor: y = 2.5 (original position)
    plt.text(-0.15, 4, L("Hypomethylated in Tumor"),
             ha="right", va="top", fontsize=8, color=color_tumor)
    plt.text( 0.15, 4, L("Hypermethylated in Tumor"),
             ha="left",  va="top", fontsize=8, color=color_tumor)

    # Adjacent: y = 5.0 (original position)
    plt.text(-0.05, 8, L("Hypomethylated in Adjacent"),
             ha="right", va="top", fontsize=8, color=color_adjacent)
    plt.text( 0.05, 8, L("Hypermethylated in Adjacent"),
             ha="left",  va="top", fontsize=8, color=color_adjacent)

    # ----------------------------------------------------------
    place_legend(position="upper right")

    dataset = globals().get("DATASET_NAME", "DATASET")
    fig_idx = globals().get("PLOT_INDEX", 5)
    savefig(f"delta_beta_overlaid")

    return results

# =====================
# INTRAGROUP VARIABILITY & CORRELATION
# =====================

def intragroup_variance_and_correlation(beta: pd.DataFrame, labels: pd.Series):
    """
    Compute and visualize:
      1. CpG-wise variance distributions by group (KDE)
      2. Sample-sample Pearson correlation heatmap
    Both plots use thesis style, Viridis colormap, full-figure border, and consistent layout.
    """
    # ====================================================
    # 1) CpG-wise variance KDE per gruppo
    # ====================================================
    var_by_group: dict[str, np.ndarray] = {}

    for g in GROUP_ORDER:
        idx = labels[labels == g].index
        if len(idx) < 2:
            continue
        group_beta = beta.loc[idx]
        v = group_beta.var(axis=0, ddof=1)
        v = v[np.isfinite(v)]
        if v.size == 0:
            continue
        var_by_group[g] = v.values if isinstance(v, pd.Series) else np.asarray(v)

    if var_by_group:
        fig = new_fig()
        ax = plt.gca()

        for g in GROUP_ORDER:
            if g not in var_by_group:
                continue
            vals = var_by_group[g]
            sns.kdeplot(
                vals,
                linewidth=1.5,
                fill=True,
                alpha=0.35,
                color=GROUP_COLORS[g],
                label=L(legend_label(g)),
                ax=ax,
            )

        ax.set_xlabel(L("CpG-wise variance of $\\beta$"))
        ax.set_ylabel(L("Density"))
        ax.set_xlim(left=0.0)

        place_legend(ax=ax, position="upper right")

        plot_name = "intragroup_variance"
        savefig(plot_name)

    # ====================================================
    # 2) Sample correlation heatmap — stile tesi + legenda custom
    # ====================================================
    var_all = beta.var(axis=0, ddof=1)
    keep = var_all.nlargest(min(PCA_TOP_CPGS, var_all.size)).index
    beta_sub = beta[keep]
    corr = beta_sub.transpose().corr(method="pearson")
    (OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
    corr.to_csv(OUTPUT_DIR / "tables" / "sample_correlation_matrix.csv")

    import matplotlib as mpl
    from matplotlib.patches import Rectangle

    fig = new_fig()
    ax = plt.gca()

    # --- Heatmap principale (Samples × Samples)
    sns.heatmap(
        corr,
        ax=ax,
        cmap="viridis",
        center=0.0,
        cbar=False,           # niente colorbar automatica
        xticklabels=False,
        yticklabels=False,
        vmin=-1.0,
        vmax=1.0,
        rasterized=True,
    )

    ax.set_xlabel(L("Samples"))
    ax.set_ylabel(L("Samples"))

    # --- Ticks numerici 1..N (coarse)
    N = corr.shape[0]

    def _nice_ticks(n_items: int, max_ticks: int = 20):
        if n_items <= 0:
            return [], []
        n_ticks = min(max_ticks, n_items)
        pos = np.linspace(0, n_items - 1, num=n_ticks, dtype=int)
        lab = (pos + 1).astype(int)
        return pos + 0.5, lab

    xlocs, xlabels = _nice_ticks(N, 20)
    ylocs, ylabels = _nice_ticks(N, 20)
    ax.set_xticks(xlocs)
    ax.set_yticks(ylocs)
    ax.set_xticklabels([str(x) for x in xlabels], fontsize=8)
    ax.set_yticklabels([str(y) for y in ylabels], fontsize=8)

    # --- Bordo attorno all'axes principale
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.9)
        spine.set_edgecolor("black")

    # ====================================================
    # 2b) Mini-legend box stile heatmap β, ma per la correlazione
    # ====================================================
    # Dimensioni del box (in coordinate figura 0–1)
    margin = 0.01
    new_w = 0.18    # larghezza del box
    new_h = 0.06    # altezza del box

    # tua scelta finale:
    new_x = 1.0 - new_w - margin - 0.03   # molto a destra
    new_y = 1.0 - new_h - margin - 0.04   # bello in alto

    cax = fig.add_axes([new_x, new_y, new_w, new_h])

    edge_col = mpl.rcParams.get("legend.edgecolor", "#D0D0D0")
    face_col = mpl.rcParams.get("legend.facecolor", "white")

    cax.set_facecolor(face_col)
    for spine in cax.spines.values():
        spine.set_visible(False)

    cax.set_xlim(0.0, 1.0)
    cax.set_ylim(0.0, 1.0)
    cax.set_xticks([])
    cax.set_yticks([])
    cax.tick_params(axis="both", bottom=False, top=False,
                    left=False, right=False, length=0)

    # --- Barra viridis orizzontale (tutta la larghezza del box)
    gradient = np.linspace(0.0, 1.0, 256).reshape(1, -1)
    cax.imshow(
        gradient,
        aspect="auto",
        cmap="viridis",
        origin="lower",
        extent=[0.0, 1.0, 0.45, 1.0],   # parte alta del box
    )

    font_size = mpl.rcParams.get("legend.fontsize", 9.5)
    font_color = "black"

    # Nessuna label tipo "Pearson correlation": solo numeri -1, 0, 1
    cax.text(
        0.0,
        0.18,
        r"$-1$",
        fontsize=font_size,
        color=font_color,
        ha="left",
        va="center",
        transform=cax.transAxes,
    )
    cax.text(
        0.5,
        0.18,
        r"$0$",
        fontsize=font_size,
        color=font_color,
        ha="center",
        va="center",
        transform=cax.transAxes,
    )
    cax.text(
        1.0,
        0.18,
        r"$1$",
        fontsize=font_size,
        color=font_color,
        ha="right",
        va="center",
        transform=cax.transAxes,
    )

    cax.set_xlabel("")
    cax.set_ylabel("")


    # --- Save figure
    plot_name = "sample_correlation_heatmap"
    savefig(plot_name)


# =====================
# DISTRIBUTIONAL SHAPE (SKEWNESS / KURTOSIS)
# =====================

def skewness_kurtosis_analysis(beta: pd.DataFrame, labels: pd.Series) -> dict:
    skews, kurts, groups = [], [], []
    for sid in beta.index:
        vals = beta.loc[sid].to_numpy(dtype=np.float32); vals = vals[np.isfinite(vals)]
        if vals.size == 0: continue
        sk = stats.skew(vals, bias=False); kt = stats.kurtosis(vals, bias=False)
        skews.append(sk); kurts.append(kt); groups.append(labels.loc[sid])
    df = pd.DataFrame({"id_tissue": beta.index[:len(skews)], "skewness": skews, "kurtosis": kurts,
                       "group": groups})
    df["legend_group"] = df["group"].map(legend_label)
    df = df.dropna(subset=["group"])
    df.to_csv(OUTPUT_DIR/"tables"/"skewness_kurtosis_per_sample.csv", index=False)
    fig = new_fig()
    palette = [GROUP_COLORS[g] for g in GROUP_ORDER if (df["group"]==g).any()]
    sns.scatterplot(data=df, x="skewness", y="kurtosis", hue="legend_group", hue_order=LEGEND_ORDER,
                    palette=palette, s=30, alpha=0.8)
    plt.xlabel(L("Skewness of $\\beta$ distribution")); plt.ylabel(L("Kurtosis of $\\beta$ distribution"))
    place_legend(position="upper right"); savefig("skewness_kurtosis")
    summary = {}
    for g in GROUP_ORDER:
        df_g = df[df["group"] == g]
        if df_g.empty: continue
        summary[g] = {"median_skewness": float(df_g["skewness"].median()), "median_kurtosis": float(df_g["kurtosis"].median())}
    save_json(summary, OUTPUT_DIR/"tables"/"skewness_kurtosis_summary.json")
    return summary

# =====================
# EMBEDDINGS (PCA / TSNE / UMAP — robust UMAP)
# =====================

def to_M_values(beta:pd.DataFrame) -> pd.DataFrame:
    B = beta.clip(lower=1e-6, upper=1-1e-6)
    return np.log2(B/(1-B))

def _prepare_umap_patch():
    """
    If sklearn.check_array doesn't accept 'ensure_all_finite', patch it for umap-learn compatibility.
    """
    try:
        import umap  # noqa
    except Exception as e:
        warn(f"UMAP not available: {e}")
        return None
    try:
        sig = inspect.signature(skval.check_array)
        if "ensure_all_finite" not in sig.parameters:
            orig = skval.check_array
            def check_array_compat(*args, **kwargs):
                kwargs.pop("ensure_all_finite", None)
                return orig(*args, **kwargs)
            skval.check_array = check_array_compat
            info("Patched sklearn.check_array to ignore 'ensure_all_finite' for UMAP.")
    except Exception as e:
        warn(f"Could not inspect/patch check_array: {e}")
    import umap as umap_mod
    return umap_mod.UMAP

def _prepare_umap_robust():
    """
    Return a UMAP class with a robust patch for sklearn.check_array
    to drop the 'ensure_all_finite' kwarg used in some umap+sklearn combos.
    """
    try:
        import umap
        import umap.umap_ as umap_mod
        from sklearn.utils import validation as skval
    except Exception as e:
        warn(f"UMAP not available or incompatible: {type(e).__name__}: {e}")
        return None

    orig_check_array = skval.check_array

    def _check_array_compat(*args, **kwargs):
        # rimuovi il kwarg incriminato se presente
        kwargs.pop("ensure_all_finite", None)
        return orig_check_array(*args, **kwargs)

    # Patch in sklearn
    skval.check_array = _check_array_compat

    # Patch anche nel modulo UMAP stesso (questo è il punto che prima mancava)
    if hasattr(umap_mod, "check_array"):
        umap_mod.check_array = _check_array_compat
    if hasattr(umap_mod, "sklearn_check_array"):
        umap_mod.sklearn_check_array = _check_array_compat

    info("Patched sklearn + umap check_array to ignore 'ensure_all_finite' for UMAP.")
    return umap.UMAP


def pca_and_embeddings(beta: pd.DataFrame, labels: pd.Series) -> dict:
    metrics = {}

    # =========
    # PREP
    # =========
    var = beta.var(axis=0, ddof=1)
    keep = var.nlargest(min(PCA_TOP_CPGS, var.size)).index
    X = to_M_values(beta.loc[:, keep])
    X = X - X.mean(axis=0)

    # =========
    # PCA
    # =========
    pca = PCA(n_components=2, random_state=SEED)
    Xp = pca.fit_transform(X)
    dfp = pd.DataFrame(Xp, columns=["PC1", "PC2"], index=beta.index)
    dfp["group"] = labels
    dfp["legend_group"] = dfp["group"].map(legend_label)
    dfp.to_csv(OUTPUT_DIR / "tables" / "pca_pc12_coords.csv")

    fig = new_fig()
    sns.scatterplot(
        data=dfp,
        x="PC1",
        y="PC2",
        hue="legend_group",
        hue_order=LEGEND_ORDER,
        palette=[GROUP_COLORS[g] for g in GROUP_ORDER if (dfp["group"] == g).any()],
        s=40,
        alpha=0.9,
    )
    plt.xlabel(L("Principal Component 1"))
    plt.ylabel(L("Principal Component 2"))
    place_legend(position="upper right")
    savefig("pca")

    ev = getattr(pca, "explained_variance_ratio_", None)
    if ev is not None:
        metrics["pca_pc1_var"] = float(ev[0])
        metrics["pca_pc2_var"] = float(ev[1])
        metrics["pca_pc1_pc2_sum"] = float(ev[:2].sum())
        info(f"PCA var expl.: PC1={ev[0]:.3f}, PC2={ev[1]:.3f}, sum2={ev[:2].sum():.3f}")

    # =========
    # t-SNE
    # =========
    ipca = IncrementalPCA(n_components=IPCA_COMPONENTS, batch_size=5000)
    Xp_ipca = ipca.fit_transform(X)

    tsne = TSNE(
        n_components=2,
        init="pca",
        perplexity=30,
        learning_rate=300,
        early_exaggeration=12,
        n_iter=1500,
        metric="euclidean",
        random_state=SEED,
    )
    Xt = tsne.fit_transform(Xp_ipca)
    dft = pd.DataFrame(Xt, columns=["tSNE1", "tSNE2"], index=beta.index)
    dft["group"] = labels
    dft["legend_group"] = dft["group"].map(legend_label)
    dft.to_csv(OUTPUT_DIR / "tables" / "tsne_coords.csv")

    fig = new_fig()
    sns.scatterplot(
        data=dft,
        x="tSNE1",
        y="tSNE2",
        hue="legend_group",
        hue_order=LEGEND_ORDER,
        palette=[GROUP_COLORS[g] for g in GROUP_ORDER if (dft["group"] == g).any()],
        s=40,
        alpha=0.9,
    )
    plt.xlabel(L("t-Distributed Stochastic Neighbor Embedding 1"))
    plt.ylabel(L("t-Distributed Stochastic Neighbor Embedding 2"))
    place_legend(position="lower right")
    savefig("tsne")

        # UMAP (robust)
    UMAP_cls = _prepare_umap_robust()
    if UMAP_cls is not None:
        try:
            reducer = UMAP_cls(
                n_neighbors=30,
                min_dist=0.1,
                metric="euclidean",
                random_state=SEED,
            )
            Xu = reducer.fit_transform(np.asarray(X, dtype=np.float32, order="C"))

            dfu = pd.DataFrame(Xu, columns=["UMAP1", "UMAP2"], index=beta.index)
            dfu["group"] = labels
            dfu["legend_group"] = dfu["group"].map(legend_label)
            dfu.to_csv(OUTPUT_DIR / "tables" / "umap_coords.csv")

            fig = new_fig()
            sns.scatterplot(
                data=dfu,
                x="UMAP1",
                y="UMAP2",
                hue="legend_group",
                hue_order=LEGEND_ORDER,
                palette=[GROUP_COLORS[g] for g in GROUP_ORDER if (dfu["group"] == g).any()],
                s=40,
                alpha=0.9,
            )
            plt.xlabel(L("UMAP1"))
            plt.ylabel(L("UMAP2"))
            place_legend(position="lower right")
            savefig("umap")

            metrics["umap_available"] = True
        except Exception as e:
            warn(f"UMAP failed or not available: {type(e).__name__}: {e}; UMAP plot skipped.")
            metrics["umap_available"] = False
    else:
        warn("UMAP not available in environment; plot skipped.")
        metrics["umap_available"] = False

    return metrics

# =====================
# LIGHTWEIGHT GLOBAL TESTS (SCREENING)
# =====================

def cliffs_delta(a:np.ndarray, b:np.ndarray) -> float:
    a = np.asarray(a); b = np.asarray(b)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size==0 or b.size==0: return np.nan
    ranks = stats.rankdata(np.concatenate([a,b]))
    ra = ranks[:len(a)]; rb = ranks[len(a):]
    m, n = len(a), len(b)
    delta = (np.sum(ra) - m*(m+1)/2 - np.sum(rb) + n*(n+1)/2) / (m*n)
    return float(delta)

def screening_tests(beta:pd.DataFrame, labels:pd.Series):
    results = {}; lines = []
    for g1 in GROUP_ORDER:
        for g2 in GROUP_ORDER:
            if GROUP_ORDER.index(g2) <= GROUP_ORDER.index(g1): continue
            idx1 = labels[labels==g1].index; idx2 = labels[labels==g2].index
            if len(idx1)==0 or len(idx2)==0: continue
            a = beta.loc[idx1].to_numpy().ravel(); a = a[np.isfinite(a)]
            b = beta.loc[idx2].to_numpy().ravel(); b = b[np.isfinite(b)]
            ks_stat, ks_p = stats.ks_2samp(a, b, alternative='two-sided', method='asymp')
            ma = beta.loc[idx1].mean(axis=1).to_numpy(); mb = beta.loc[idx2].mean(axis=1).to_numpy()
            lev_stat, lev_p = stats.levene(ma, mb, center='median')
            cd = cliffs_delta(ma, mb)
            key = f"{g1}_vs_{g2}"
            results[key] = {"ks_stat":float(ks_stat), "ks_p":float(ks_p),
                            "levene_stat":float(lev_stat), "levene_p":float(lev_p),
                            "cliffs_delta_on_means":float(cd)}
            lines.append(f"{g1}_vs_{g2}: KS p={ks_p:.2e}, Levene p={lev_p:.2e}, Cliff's δ={cd:.3f}")
    save_json(results, OUTPUT_DIR/"tables"/"screening_tests.json")
    if lines: info("Screening tests summary:" + "".join(["\n  - "+l for l in lines]))
    else: warn("Screening tests: not enough groups to compare")
    return results

# =====================
# ROBUST MANIFEST LOADER + COVERAGE
# =====================

def detect_separator(sample_line: str, default=","):
    if "\t" in sample_line: return "\t"
    if ";" in sample_line:  return ";"
    if "," in sample_line:  return ","
    return default

def open_text_auto(path: str):
    if path.endswith(".gz"):  return gzip.open(path, "rt", errors="ignore", newline="")
    if path.endswith(".bz2"): return bz2.open(path, "rt", errors="ignore", newline="")
    return open(path, "rt", errors="ignore", newline="")

def auto_discover_manifest() -> Path | None:
    patterns = [
        "/kaggle/input/**/*EPIC*.csv*",
        "/kaggle/input/**/*850*.csv*",
        "/kaggle/input/**/*450*.csv*",
        "/kaggle/input/**/*.csv*",
        "/kaggle/input/**/*.tsv*",
        "/kaggle/input/**/*.txt*",
    ]
    cands = []
    for p in patterns: cands.extend(glob.glob(p, recursive=True))
    if not cands: return None
    def rank(p):
        s = os.path.basename(p).lower(); r = 0
        if "epic" in s or "850" in s: r += 100
        if "450" in s or "hm450" in s: r += 50
        if s.endswith(".csv") or s.endswith(".csv.gz") or s.endswith(".csv.bz2"): r += 10
        if "manifest" in s: r += 5
        return -r
    cands.sort(key=rank)
    for c in cands:
        try:
            if os.path.getsize(c) > 0: return Path(c)
        except Exception: continue
    return Path(cands[0])

def load_illumina_manifest_any(path: Path,
                               header_probe_keys=("IlmnID","Probe_ID","Name","TargetID","ID_REF"),
                               fallback_skiprows=6) -> pd.DataFrame:
    """
    Robust manifest loader:
    - tries to find [Assay] and header line; if not found, fallback skiprows=6
    - reads everything with pandas dtype=str
    """
    with open_text_auto(str(path)) as f:
        lines = f.readlines()
    if len(lines) < 8: raise ValueError("Manifest file too short to contain header + data.")

    header_idx = None
    # 1) Look for [Assay]
    for i, line in enumerate(lines[:200]):
        if line.strip().lower().startswith("[assay]"):
            for j in range(i+1, min(i+50, len(lines))):
                s = lines[j].strip()
                if s and not s.startswith("["):
                    header_idx = j; break
            break
    # 2) Fallback: look for known header tokens
    if header_idx is None:
        for i, line in enumerate(lines[:50]):
            s = line.strip()
            parts = [x.strip() for x in s.split(",")]
            if len(parts)==1: parts = [x.strip() for x in s.split("\t")]
            if len(parts)==1: parts = [x.strip() for x in s.split(";")]
            if any(k in parts for k in header_probe_keys):
                header_idx = i; break
    # 3) Ultimate fallback: skip first 6 lines
    if header_idx is None:
        header_idx = fallback_skiprows
        warn(f"[coverage] Header not detected → applying fallback skiprows={fallback_skiprows}.")

    header_line = lines[header_idx].strip()
    sep = detect_separator(header_line, default=",")
    csv_text = "".join(lines[header_idx:])
    df = pd.read_csv(io.StringIO(csv_text), sep=sep, dtype=str, low_memory=False)

    # Normalize CpG column name
    for cand in ("IlmnID","Probe_ID","Name","TargetID","ID_REF","CpG"):
        if cand in df.columns:
            df = df.rename(columns={cand: "CpG"})
            break
    if "CpG" not in df.columns:
        raise ValueError("CpG column not found (expected IlmnID/Probe_ID/Name/TargetID/ID_REF).")
    return df

def coverage_with_manifest_robust(beta: pd.DataFrame, manifest_path: Path) -> dict:
    if manifest_path is None or (not manifest_path.exists()):
        ad = auto_discover_manifest()
        if ad is None:
            warn("[coverage] Manifest not found — skipping coverage.")
            return {}
        manifest_path = ad
        info(f"[coverage] Auto-discovered manifest: {manifest_path}")

    info("Coverage via manifest (robust)…")
    ann_full = load_illumina_manifest_any(manifest_path)

    beta_cpgs = set(map(str, beta.columns))
    if "CpG" not in ann_full.columns:
        warn("[coverage] No 'CpG' column after parsing — skipping coverage.")
        return {}

    ann = ann_full[ann_full["CpG"].isin(beta_cpgs)].copy()
    if ann.empty:
        warn("[coverage] No manifest CpGs matched the dataset columns — skipping coverage.")
        return {}

    mani_cpgs = set(ann["CpG"])
    inter = beta_cpgs & mani_cpgs
    coverage_summary = {
        "n_beta_cpgs": len(beta_cpgs),
        "n_manifest_cpgs": len(mani_cpgs),
        "n_intersection": len(inter),
        "pct_covered": (100.0 * len(inter) / len(beta_cpgs)) if beta_cpgs else 0.0,
    }
    cov_lines = [
        "Manifest coverage summary",
        f"- CpG in β-matrix: {coverage_summary['n_beta_cpgs']:,}",
        f"- CpG in manifest: {coverage_summary['n_manifest_cpgs']:,}",
        f"- Intersection:    {coverage_summary['n_intersection']:,} ({coverage_summary['pct_covered']:.2f}%)",
        "",
    ]
    (OUTPUT_DIR / "coverage_manifest.txt").write_text("\n".join(cov_lines), encoding="utf-8")
    ok("Coverage summary\n" + "\n".join(cov_lines))

    # Genes
    gene_counts = None
    if "UCSC_RefGene_Name" in ann.columns:
        gn = (
            ann["UCSC_RefGene_Name"]
            .astype(str)
            .replace({"nan": np.nan})
            .dropna()
            .str.replace(";", ",")
            .str.split(",")
            .explode()
            .str.strip()
        )
        gn = gn[gn != ""]
        if not gn.empty:
            gene_counts = gn.value_counts()
            coverage_summary["n_genes"] = int(gene_counts.index.nunique())
            (OUTPUT_DIR / "tables" / f"{DATASET_NAME}_genes_present_with_cpg_counts.csv").write_text(
                gene_counts.to_csv(header=["n_cpgs"])
            )
            with open(OUTPUT_DIR / "tables" / f"{DATASET_NAME}_genes_present.txt", "w", encoding="utf-8") as f:
                for g in gene_counts.index:
                    f.write(f"{g}\n")
            ok(f"Genes present written ({coverage_summary['n_genes']} unique genes).")

    # Coverage by gene region
    if "UCSC_RefGene_Group" in ann.columns:
        reg_counts = ann["UCSC_RefGene_Group"].fillna("NA").value_counts().sort_values(ascending=False)
        coverage_summary["coverage_by_region"] = {str(k): int(v) for k, v in reg_counts.items()}
        (OUTPUT_DIR / "tables" / f"{DATASET_NAME}_coverage_by_gene_region.csv").write_text(
            reg_counts.to_csv(header=["n_cpgs"])
        )

    # Coverage by CpG island context
    if "Relation_to_UCSC_CpG_Island" in ann.columns:
        ctx_counts = ann["Relation_to_UCSC_CpG_Island"].fillna("NA").value_counts().sort_values(ascending=False)
        coverage_summary["coverage_by_cpg_context"] = {str(k): int(v) for k, v in ctx_counts.items()}
        (OUTPUT_DIR / "tables" / f"{DATASET_NAME}_coverage_by_cpg_context.csv").write_text(
            ctx_counts.to_csv(header=["n_cpgs"])
        )

        # ---- PLOT 1: barplot con legenda per ogni categoria e x-ticks dritti ----
        fig = new_fig()
        ax = plt.gca()

        # costruiamo un DataFrame per usare hue=context
        df_ctx = ctx_counts.reset_index()
        df_ctx.columns = ["context", "n_cpgs"]

        sns.barplot(
            data=df_ctx,
            x="context",
            y="n_cpgs",
            hue="context",        # così seaborn crea una legend per ogni categoria
            dodge=False,
            palette="viridis",
            ax=ax,
        )

        # etichette asse x dritte
        plt.xticks(rotation=0, ha="center")
        plt.xlabel("CpG Context")
        plt.ylabel(L("Number of CpGs"))

        # la legenda ora contiene tutte le categorie (NA, Island, Shore, …)
        # place_legend riusa gli handle/label correnti e solo li sposta/formata
        place_legend(ax=ax, position="upper right")

        # opzionale: togli il titolo "context" dalla legenda se non lo vuoi
        leg = ax.get_legend()
        if leg is not None:
            leg.set_title("")

        savefig("coverage_by_cpg_context")

    # Dynamic network degree proxy (log–log)
    if gene_counts is not None and not gene_counts.empty:
        degrees = gene_counts.sort_values(ascending=False).astype(float).values
        ranks = np.arange(1, len(degrees) + 1, dtype=float)
        fig = new_fig()

        # ---- PLOT 2: scatter con label direttamente sullo scatter ----
        plt.scatter(
            np.log10(ranks),
            np.log10(degrees),
            s=8,
            alpha=0.7,
            color=VIRIDIS(0.6),
            label=L("Gene-level CpG degree (proxy for network node degree)"),
        )
        plt.xlabel(L("log$_{10}$(rank)"))
        plt.ylabel(L("log$_{10}$(CpGs per gene)"))

        # Ora la legenda mostra il PALLINO verde + testo
        place_legend(position="upper right")
        savefig("dynamic_network_degree_proxy")

        coverage_summary["dynamic_network_proxy"] = True
    else:
        coverage_summary["dynamic_network_proxy"] = False

    ok("Manifest coverage analysis complete.")
    return coverage_summary


# =====================
# DNB-LIKE NETWORK PLOT (overall, all samples)
# =====================

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def _select_top_dnb(beta: pd.DataFrame, labels: pd.Series, group_a: str, group_b: str, n_dnb_cpg: int):
    """Return (dnb_cpg_index, delta_series) where delta = mean(group_a) - mean(group_b)."""
    idx_a = labels[labels == group_a].index
    idx_b = labels[labels == group_b].index
    if len(idx_a) == 0 or len(idx_b) == 0:
        return pd.Index([]), pd.Series(dtype=float)
    mean_a = beta.loc[idx_a].mean(axis=0)
    mean_b = beta.loc[idx_b].mean(axis=0)
    delta  = (mean_a - mean_b).dropna()
    dnb    = delta.abs().nlargest(n_dnb_cpg).index
    return dnb, delta

def _edge_widths_alphas(weights: np.ndarray):
    """Map |corr| in [0,1] to visually pleasant widths/alphas."""
    w = np.clip(weights, 0.0, 1.0)
    widths = 0.4 + 2.2 * w
    alphas = 0.08 + 0.42 * w
    return widths, alphas

def _full_figure_border(fig):
    border = Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                       fill=False, linewidth=1.0, edgecolor="black",
                       zorder=1000, clip_on=False)
    fig.add_artist(border)

from warnings import warn

def dnb_network_global(
    beta: pd.DataFrame,
    labels: pd.Series,
    group_a: str,
    group_b: str,
    n_dnb_cpg: int = 200,
    corr_threshold: float = 0.8,
):
    """
    Dynamic-network-like CpG graph using ALL samples.
    - Nodes = top-|Δβ| CpGs
    - Edges = |corr| >= corr_threshold
    - Node color = Δβ (group_a - group_b)
    - Node size = degree

    Layout: spring layout + light radial compression of outer nodes.
    Legend bottom-left with node degree and Δβ colormap.
    """
    THESIS_BG = "white"

    # 1) Select CpGs by |Δβ|
    dnb_cpgs, delta = _select_top_dnb(beta, labels, group_a, group_b, n_dnb_cpg)
    if len(dnb_cpgs) == 0:
        warn(f"[DNB] Missing Samples for groups '{group_a}' or '{group_b}'.")
        return

    # 2) Correlations on ALL samples
    B = beta[dnb_cpgs].copy()
    B = B.apply(lambda x: (x - x.mean()) / x.std(ddof=1), axis=0)
    B = B.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    C = B.corr(method="pearson").to_numpy()
    cols = list(dnb_cpgs)

    # 3) Graph
    G = nx.Graph()
    for c in cols:
        G.add_node(c, delta=float(delta[c]))
    n = len(cols)
    for i in range(n):
        for j in range(i + 1, n):
            r = C[i, j]
            if np.isfinite(r) and abs(r) >= corr_threshold:
                G.add_edge(cols[i], cols[j], weight=float(abs(r)))
    if G.number_of_edges() == 0:
        warn(f"[DNB] No edges above {corr_threshold}.")
        return

    # 4) Layout – k "normale", poi solo outlier compressi radialmente
    pos = nx.spring_layout(G, seed=42, k=0.15, iterations=250)

    # compressione radiale leggera degli outlier (nodi troppo lontani)
    node_names = list(G.nodes())
    xy = np.array([pos[n] for n in node_names])
    center = xy.mean(axis=0)
    diffs = xy - center
    radii = np.linalg.norm(diffs, axis=1)

    if np.any(np.isfinite(radii)) and radii.max() > 0:
        r0 = np.nanpercentile(radii, 70)  # entro questo r non tocco
        alpha = 0.4                       # quanto comprimere la coda
        new_xy = xy.copy()
        for i, r in enumerate(radii):
            if r > r0:
                # nuovo r più vicino al centro
                new_r = r0 + alpha * (r - r0)
                if r > 0:
                    new_xy[i] = center + (new_r / r) * diffs[i]
        # aggiorna pos
        for name, coord in zip(node_names, new_xy):
            pos[name] = coord

    # 5) Visual encodings
    deltas = np.array([G.nodes[n]["delta"] for n in G.nodes()])
    vmax = float(np.nanmax(np.abs(deltas))) if np.isfinite(np.nanmax(np.abs(deltas))) else 1.0
    if vmax == 0:
        vmax = 1.0
    norm = mpl.colors.Normalize(vmin=-vmax, vmax=+vmax)
    cmap = VIRIDIS

    degrees = np.array([G.degree(n) for n in G.nodes()])
    if degrees.max() == 0:
        sizes = np.full_like(degrees, 40.0, dtype=float)
    else:
        sizes = 25.0 + 90.0 * (degrees / degrees.max())

    # 6) Figure – thesis style
    fig = new_fig()
    fig.patch.set_facecolor(THESIS_BG)
    ax = fig.add_subplot(111)

    apply_thesis_fn = globals().get("apply_thesis", None)
    if apply_thesis_fn is not None:
        try:
            apply_thesis_fn(ax=ax, legend=False)
        except TypeError:
            try:
                apply_thesis_fn(ax, legend=False)
            except TypeError:
                apply_thesis_fn(ax)

    ax.set_xticks([])
    ax.set_yticks([])

    # --- Edges sotto ---
    edge_list = list(G.edges(data=True))
    if edge_list:
        ws = np.array([e[2].get("weight", 0.0) for e in edge_list])
        w, a = _edge_widths_alphas(ws)
        for (u, v, d), wi, ai in zip(edge_list, w, a):
            ax.plot(
                [pos[u][0], pos[v][0]],
                [pos[u][1], pos[v][1]],
                linewidth=wi,
                color="grey",
                alpha=ai,
                zorder=1,
            )

    # --- Nodes sopra ---
    node_xy = np.array([pos[n] for n in G.nodes()])
    sc = ax.scatter(
        node_xy[:, 0],
        node_xy[:, 1],
        s=sizes,
        c=deltas,
        cmap=cmap,
        norm=norm,
        edgecolors="black",
        linewidths=0.3,
        zorder=2,
    )

    # ================================================
    #           LEGEND MODERNA (bottom-left)
    # ================================================
    # mooolto più a sinistra
    legend_ax = fig.add_axes([0.03, 0.05, 0.30, 0.24])  # [x0, y0, w, h]
    legend_ax.set_xlim(0.0, 1.0)
    legend_ax.set_ylim(0.0, 1.0)
    legend_ax.axis("off")
    legend_ax.set_facecolor("white")
    legend_ax.patch.set_alpha(1.0)  # <-- sfondo bianco pulito


    # Bordo stile tesi intorno alla legenda
    legend_edge_col = mpl.rcParams.get("legend.edgecolor", "#D0D0D0")
    # --- SFONDO PIENO PRIMA DI TUTTO ---
    bg = Rectangle(
        (0, 0),
        1,
        1,
        transform=legend_ax.transAxes,
        fill=True,
        facecolor="white",
        edgecolor="none",
        alpha=1.0,
        zorder=0
    )
    legend_ax.add_patch(bg)

# (colormap, puntini, testi verranno disegnati sopra, con zorder standard)

    # --- BORDO SOPRA IL CONTENUTO ---
    border = Rectangle(
        (0, 0),
        1,
        1,
        transform=legend_ax.transAxes,
        fill=False,
        edgecolor=legend_edge_col,
        linewidth=0.8,
        joinstyle="round",
        zorder=1000
    )
    legend_ax.add_patch(border)


    # --- Node degree: 3 pallini pieni, colore viridis ---
    deg_values = np.unique(degrees)
    deg_values = deg_values[deg_values > 0]
    if deg_values.size == 0:
        deg_values = np.array([1, 2, 3])
    elif len(deg_values) > 3:
        deg_values = np.linspace(deg_values.min(), deg_values.max(), 3, dtype=int)
        deg_values = np.unique(deg_values)

    y_nodes = 0.82
    # (2) spazio dopo "Node degree" prima dei pallini
    legend_ax.text(
        0.06,
        y_nodes,
        "Node Degree",
        ha="left",
        va="center",
        fontsize=9,
    )

    # pallini spostati a destra, con un bel gap
    x_nodes = [0.50, 0.70, 0.90]
    max_deg = degrees.max() if degrees.max() > 0 else 1.0
    color_ref = cmap(0.75)  # colore medio viridis

    for x, d_ref in zip(x_nodes, deg_values):
        size_ref = 25.0 + 90.0 * (d_ref / max_deg)
        legend_ax.scatter(
            x,
            y_nodes,
            s=size_ref,
            facecolor=color_ref,
            edgecolor="none",     # pieni, senza bordo
        )
        legend_ax.text(
            x,
            y_nodes - 0.17,
            rf"{int(d_ref)}",
            ha="center",
            va="center",
            fontsize=8,
        )

    # --- Colormap piccola orizzontale con numeri sotto ---
    gradient = np.linspace(0.0, 1.0, 256).reshape(1, -1)
    legend_ax.imshow(
        gradient,
        aspect="auto",
        cmap=cmap,
        origin="lower",
        extent=[0.10, 0.90, 0.14, 0.40],  # barra in basso, bella alta
    )

    font_size = mpl.rcParams.get("legend.fontsize", 8.0)
    font_color = "black"

    delta_min = float(np.nanmin(deltas)) if np.isfinite(np.nanmin(deltas)) else -vmax
    delta_max = float(np.nanmax(deltas)) if np.isfinite(np.nanmax(deltas)) else vmax

    legend_ax.text(
        0.10,
        0.10,
        f"{delta_min:.2f}",
        ha="left",
        va="top",
        fontsize=font_size,
        color=font_color,
    )
    legend_ax.text(
        0.50,
        0.10,
        r"$0$",
        ha="center",
        va="top",
        fontsize=font_size,
        color=font_color,
    )
    legend_ax.text(
        0.90,
        0.10,
        f"{delta_max:.2f}",
        ha="right",
        va="top",
        fontsize=font_size,
        color=font_color,
    )

    # (3) label Δβ centrata rispetto alla colormap (che va da x=0.10 a x=0.90)
    legend_ax.text(
        0.50,
        0.48,
        rf"$\Delta\beta$ ({group_a} - {group_b})",
        ha="center",
        va="center",
        fontsize=9,
    )

    # 7) Save fig
    dataset = globals().get("DATASET_NAME", "DATASET")
    fig_idx = globals().get("PLOT_INDEX", 9)
    safe_name = f"dnb_network_{group_a.lower()}_minus_{group_b.lower()}"
    savefig(safe_name)


def dnb_network_two_contrasts(beta: pd.DataFrame, labels: pd.Series,
                              n_dnb_cpg: int = 200, corr_threshold: float = 0.8):
    """
    Convenience wrapper to produce the two requested global networks:
    1) Tumor − Normal
    2) Adjacent − Normal
    """
    
    # 1) Tumor − Normal
    dnb_network_global(beta, labels, group_a="Tumor", group_b="Normal",
                       n_dnb_cpg=n_dnb_cpg, corr_threshold=corr_threshold)

    # advance plot index if you use manual numbering
    if "PLOT_INDEX" in globals():
        globals()["PLOT_INDEX"] = globals()["PLOT_INDEX"] + 1

    # 2) Adjacent − Normal
    dnb_network_global(beta, labels, group_a="Adjacent", group_b="Normal",
                       n_dnb_cpg=n_dnb_cpg, corr_threshold=corr_threshold)


# =====================
# SUMMARY PAGE & METRICS
# =====================

def build_summary_text(snapshot: dict, labels: pd.Series, delta_stats: dict,
                       skew_kurt_stats: dict, pca_metrics: dict, coverage_summary: dict) -> str:
    n_samples = snapshot.get("n_samples"); n_cpg = snapshot.get("n_cpg")
    missing_s_med = snapshot.get("missing_sample_median", float("nan"))
    missing_c_med = snapshot.get("missing_cpg_median", float("nan"))
    missing_s_iqr = snapshot.get("missing_sample_iqr", float("nan"))
    missing_c_iqr = snapshot.get("missing_cpg_iqr", float("nan"))
    group_counts = labels.value_counts(); ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    fig_dir = OUTPUT_DIR/"figures"; figs = sorted(p.name for p in fig_dir.glob("*.pdf")); n_figs = len(figs)
    lines: list[str] = []
    lines.append(f"[DATA EXPLORATION & VISUALIZATION — {DATASET_NAME}]")
    lines.append(f"Samples: {n_samples}  |  CpGs: {n_cpg}")
    lines.append("Sample composition:")
    for g in GROUP_ORDER: lines.append(f"  - {g}: {int(group_counts.get(g, 0))}")
    lines.append("β-value matrix type: float32 (Polars-derived, written to Parquet upstream)")
    lines.append(f"Output directory: {OUTPUT_DIR.resolve()}"); lines.append(f"Report generated: {ts}"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("1. DATA INTEGRITY"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append(f"- β matrix and pheno successfully aligned on id_tissue.")
    lines.append(f"- Per-sample missing (median / IQR): {missing_s_med:.4f} / {missing_s_iqr:.4f}")
    lines.append(f"- Per-CpG missing (median / IQR):   {missing_c_med:.4f} / {missing_c_iqr:.4f}")
    lines.append(f"- Flags: samples ≥{SAMPLE_NA_FLAG*100:.1f}% missing → {snapshot.get('n_samples_flagged_missing')}, "
                 f"CpGs ≥{CPG_NA_FLAG*100:.1f}% missing → {snapshot.get('n_cpg_flagged_missing')}")
    lines.append("Figures: 01_missing_per_sample, 02_missing_per_cpg"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("2. GLOBAL β DISTRIBUTIONS"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append("- Global β density by tissue group; ridge-style per group; per-sample mean β by group.")
    lines.append("Figures: 03_beta_density, 04_ridge_normal, 05_ridge_adjacent, 06_ridge_tumor, 07_mean_beta_per_sample, 08_mean_beta_distribution_by_group")
    lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("3. OUTLIER BURDEN & CPG HEATMAP"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append("- Robust outlier burden per sample (Normal as reference when available).")
    lines.append("- Heatmap of top CpGs by outlier-sample count.")
    lines.append("Figures: 09_outlier_burden, 10_heatmap_top_outlier_cpgs"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("4. DIFFERENTIAL METHYLATION (Δβ)"); lines.append("─────────────────────────────────────────────────────────────")
    if "Tumor_vs_Normal" in delta_stats:
        tn = delta_stats["Tumor_vs_Normal"]
        lines.append("- Δβ (Tumor − Normal): clear hypo-/hyper-methylated components.")
        lines.append(f"  Example summary: mean |Δβ| ≈ {tn.get('mean_abs_delta', float('nan')):.4f}.")
    else: lines.append("- Δβ (Tumor − Normal): not computed (missing group).")
    if "Adjacent_vs_Normal" in delta_stats:
        an = delta_stats["Adjacent_vs_Normal"]
        lines.append("- Δβ (Adjacent − Normal): intermediate shifts consistent with field-defect hypothesis.")
        lines.append(f"  Example summary: mean |Δβ| ≈ {an.get('mean_abs_delta', float('nan')):.4f}.")
    else: lines.append("- Δβ (Adjacent − Normal): not computed (missing group).")
    lines.append("Figures: 11_delta_beta_tumor_vs_normal, 12_delta_beta_adjacent_vs_normal"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("5. INTRAGROUP VARIABILITY & SAMPLE CORRELATION"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append("- CpG-wise variance distributions across groups; sample Pearson correlation heatmap.")
    lines.append("Figures: 13_intragroup_variance, 14_sample_correlation_heatmap"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("6. DISTRIBUTIONAL SHAPE (SKEWNESS / KURTOSIS)"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append("Figure: 15_skewness_kurtosis"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("7. DIMENSIONALITY REDUCTION (PCA / t-SNE / UMAP)"); lines.append("─────────────────────────────────────────────────────────────")
    if "pca_pc1_var" in pca_metrics:
        lines.append(f"- PCA: PC1={pca_metrics['pca_pc1_var']:.3f}, PC2={pca_metrics['pca_pc2_var']:.3f}, sum2={pca_metrics['pca_pc1_pc2_sum']:.3f}.")
    else:
        lines.append("- PCA variance explained not available.")
    lines.append("- t-SNE built on Incremental PCA.")
    lines.append("- UMAP: executed if umap-learn is available; otherwise skipped gracefully.")
    lines.append("Figures: 16_pca, 17_tsne, 18_umap (if available)"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("8. GENE ANNOTATION & CpG CONTEXT (MANIFEST-BASED)"); lines.append("─────────────────────────────────────────────────────────────")
    if coverage_summary:
        if "n_genes" in coverage_summary:
            lines.append(f"- CpG→gene mapping (subset): {coverage_summary['n_genes']} unique genes represented.")
            lines.append("- Full gene list saved (genes_present*.csv/txt).")
        if "coverage_by_cpg_context" in coverage_summary: lines.append("- CpG context distribution computed from manifest.")
        if "coverage_by_region" in coverage_summary: lines.append("- Genomic region coverage summarized from manifest.")
        if coverage_summary.get("dynamic_network_proxy", False):
            lines.append("- Log–log proxy of gene-level CpG degree saved (dynamic_network_degree_proxy).")
    else:
        lines.append("- Coverage not computed (manifest missing or incompatible).")
    lines.append("Figures: coverage_by_gene_region, coverage_by_cpg_context, dynamic_network_degree_proxy (if any)"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("9. DNB-LIKE NETWORKS"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append("- Overall DNB-like network (all samples) + per-group networks using same DNB CpGs.")
    lines.append("Figures: dnb_network (overall), dnb_network_normal / _adjacent / _tumor (if groups available)"); lines.append("")
    lines.append("─────────────────────────────────────────────────────────────")
    lines.append("10. GENERATED ARTIFACTS"); lines.append("─────────────────────────────────────────────────────────────")
    lines.append(f"- Number of figures: {n_figs}"); lines.append("- Figures:")
    for f in figs: lines.append(f"  - {f}")
    lines.append("- Tables: see 'tables/' subfolder for CSV/JSON summaries.")
    lines.append("─────────────────────────────────────────────────────────────")
    return "\n".join(lines)

def new_fig():
    """Create a figure with the thesis style already applied."""
    if "apply_thesis_style" in globals():
        apply_thesis_style()
    return plt.figure()

# =====================
# MAIN
# =====================

if __name__ == "__main__":
   # MAIN
    warnings.filterwarnings('ignore')
    seed_everything(SEED)
    ensure_dirs()
    apply_thesis_style(USE_TEX)
    
    info(f"Load β from {PATH_BETA}")
    beta = load_beta_matrix(PATH_BETA, orient=ORIENT, id_ref_col=ID_REF_COL)
    
    info(f"Load pheno from {PATH_PHENO}")
    pheno = load_pheno(PATH_PHENO)
    
    info("Align β and pheno on id_tissue…")
    beta, labels, pheno_aligned = align_beta_pheno(beta, pheno)
    ok(f"β matrix ready: shape={beta.shape}")

# 1) Snapshot & Missingness
info("Snapshot & Missingness…")
snapshot = snapshot_and_missing(beta, labels)
save_json(snapshot, OUTPUT_DIR/"tables"/"dataset_snapshot.json")
ok("Snapshot complete")

# 2) Distributions
info("Distributions: global β density, ridge plots by group, mean β…")
beta_density_random_samples(beta, labels)
beta_density_group_mean(beta, labels)
ridge_by_group(beta, labels)
mean_beta_by_sample(beta, labels)

# 3) Sample-level QC (outliers) + CpG heatmap
info("Sample-level QC: outlier burden…")
ob_df = outlier_burden(beta, labels)

info("CpG-level exploration: heatmap (top outlier CpGs)…")
OUTPUT_DIR = Path(f"./EXPLORATION_GSE287331")

cpg_outlier_heatmap_numbered(beta)

# 4) Basic contrasts (Δβ)
info("Basic contrasts: Δβ Tumor–Normal / Adjacent–Normal…")
delta_stats = delta_beta_distributions(beta, labels)

# 5) Intragroup variability & correlation
info("Intragroup variance and sample correlation…")
intragroup_variance_and_correlation(beta, labels)

# 6) Distributional shape (skewness / kurtosis)
info("Skewness / Kurtosis analysis…")
skew_kurt_stats = skewness_kurtosis_analysis(beta, labels)

# 7) Embeddings (PCA / t-SNE / robust UMAP)
info("Embeddings: PCA / t-SNE / UMAP…")
pca_metrics = pca_and_embeddings(beta, labels)

# 8) Lightweight tests (KS / Levene / Cliff's δ)
info("Screening tests: KS / Levene / Cliff's δ on group-wise distributions…")
screening_tests(beta, labels)

# 9) Manifest coverage (robust) + dynamic network proxy
info("Coverage via manifest (robust)…")
try:
    coverage_summary = coverage_with_manifest_robust(beta, PATH_MANIFEST)
except Exception as e:
    warn(f"[coverage] skipped: {e}")
    coverage_summary = {}

# 10) DNB-like network (overall, all samples)
info("Building overall DNB-like network plot (all samples)…")
dnb_overall_path = OUTPUT_DIR / "figures" / f"dnb_network.pdf"
dnb_network_two_contrasts(beta, labels, n_dnb_cpg=200, corr_threshold=0.8)

# 12) Write detailed summary.txt
info("Writing detailed summary.txt…")
summary_text = build_summary_text(
    snapshot=snapshot,
    labels=labels,
    delta_stats=delta_stats,
    skew_kurt_stats=skew_kurt_stats,
    pca_metrics=pca_metrics,
    coverage_summary=coverage_summary
)
with open(OUTPUT_DIR/"summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text + "\n")
ok("summary.txt written")

ok("Done. All artifacts saved under: " + str(OUTPUT_DIR.resolve()))